# X-Ray Baggage Anomaly Detection — YOLOv11s Training

**Project:** xRay_Detection for Saudi Ports · **Milestone:** M1.1 (baseline training)  
**Dataset:** [orvile/x-ray-baggage-anomaly-detection](https://www.kaggle.com/datasets/orvile/x-ray-baggage-anomaly-detection)  
**Baseline reference:** [killa92/map-0-9-baggage-anomaly-detection-yolov11](https://www.kaggle.com/code/killa92/map-0-9-baggage-anomaly-detection-yolov11)

## Kaggle setup (required)
1. Create a new Kaggle notebook, enable **GPU** (T4 × 2 or P100).
2. Under **Add Data** → add the dataset `orvile/x-ray-baggage-anomaly-detection` (it will mount at `/kaggle/input/x-ray-baggage-anomaly-detection/`).
3. Turn **Internet** ON (needed to pip-install Ultralytics and download YOLOv11s pretrained weights).
4. Run cells top-to-bottom.

## Goal
Train YOLOv11s on 5 classes (`gun, knife, pliers, scissors, wrench`), reach **mAP@0.5 ≥ 0.85**, export `best.pt` and `best.onnx`.

## 1. Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3.0" onnx onnxslim onnxruntime

In [ ]:
import os
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Inspect the dataset

Find the Kaggle-mounted dataset root and confirm its structure. YOLO expects:
```
<root>/
├── data.yaml              # or we create one
├── images/{train,val,test}/*.jpg
└── labels/{train,val,test}/*.txt
```

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")

# Discover dataset root (handles different folder naming)
candidates = [p for p in KAGGLE_INPUT.iterdir() if "x-ray" in p.name.lower() or "baggage" in p.name.lower()]
assert candidates, f"Dataset not mounted. Available: {list(KAGGLE_INPUT.iterdir())}"
DATASET_ROOT = candidates[0]
print(f"Dataset root: {DATASET_ROOT}")

# Walk top 2 levels
for p in sorted(DATASET_ROOT.rglob("*"))[:40]:
    depth = len(p.relative_to(DATASET_ROOT).parts)
    if depth <= 2:
        print("  " * depth + p.name + ("/" if p.is_dir() else ""))

In [ ]:
# Count images and labels per split
for split in ["train", "val", "valid", "test"]:
    for sub in ["images", "labels"]:
        p = DATASET_ROOT / sub / split
        if p.exists():
            n = sum(1 for _ in p.rglob("*.*"))
            print(f"{sub}/{split}: {n}")

# Also search for an existing data.yaml
yamls = list(DATASET_ROOT.rglob("*.yaml")) + list(DATASET_ROOT.rglob("*.yml"))
print("\nYAML files found:", yamls)

## 3. Build / adapt `data.yaml`

We write a fresh `data.yaml` to `/kaggle/working/` pointing at the mounted split folders. If the dataset ships its own `data.yaml`, we still override it to guarantee paths and class order.

In [ ]:
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)

# Canonical class order used across the project
CLASS_NAMES = ["gun", "knife", "pliers", "scissors", "wrench"]
CLASS_NAMES_AR = ["مسدس", "سكين", "كماشة", "مقص", "مفتاح ربط"]

# Resolve split directories (fall back to 'valid' if 'val' is missing — Roboflow convention)
def first_existing(*paths):
    for p in paths:
        if p.exists():
            return p
    return None

train_dir = first_existing(DATASET_ROOT / "images" / "train", DATASET_ROOT / "train" / "images")
val_dir = first_existing(DATASET_ROOT / "images" / "val", DATASET_ROOT / "images" / "valid",
                         DATASET_ROOT / "val" / "images", DATASET_ROOT / "valid" / "images")
test_dir = first_existing(DATASET_ROOT / "images" / "test", DATASET_ROOT / "test" / "images")

assert train_dir and val_dir, f"train/val dirs not found — inspect the structure above"
print(f"train: {train_dir}")
print(f"val:   {val_dir}")
print(f"test:  {test_dir}")

In [ ]:
DATA_YAML = WORK / "data.yaml"

yaml_content = f"""# xRay_Detection — YOLO dataset config
path: {DATASET_ROOT}
train: {train_dir.relative_to(DATASET_ROOT)}
val: {val_dir.relative_to(DATASET_ROOT)}
{'test: ' + str(test_dir.relative_to(DATASET_ROOT)) if test_dir else ''}

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
DATA_YAML.write_text(yaml_content)
print(DATA_YAML.read_text())

## 4. Train YOLOv11s

**Config rationale:**
- `model=yolo11s.pt` — 9.4M params, fast on T4, proven > 0.9 mAP on this dataset.
- `imgsz=640` — standard; X-rays often come at larger resolution but 640 is plenty for 5 coarse classes.
- `epochs=100`, `patience=20` — early stop kicks in if val plateaus.
- `optimizer=AdamW`, `lr0=1e-3` — stable for transfer learning.
- **`hsv_h=hsv_s=hsv_v=0`** — X-ray pseudo-color encodes material (organic/metal). We must NOT jitter hue/saturation; that would corrupt the signal. This is the main deviation from generic YOLO recipes.
- `mosaic=0.5` — X-rays have less spatial regularity than natural images; heavy mosaic can create nonsensical compositions.
- `close_mosaic=10` — disable mosaic for last 10 epochs to stabilize final mAP.

In [ ]:
model = YOLO("yolo11s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=1e-3,
    patience=20,
    # X-ray-specific augmentation tuning
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,
    close_mosaic=10,
    # Output
    project=str(WORK / "runs"),
    name="yolo11s_xray",
    exist_ok=True,
    plots=True,
    verbose=True,
)

## 5. Evaluate on validation split

In [ ]:
RUN_DIR = WORK / "runs" / "yolo11s_xray"
BEST_PT = RUN_DIR / "weights" / "best.pt"
print(f"Best weights: {BEST_PT}")

best = YOLO(str(BEST_PT))
metrics = best.val(data=str(DATA_YAML), imgsz=640, batch=16)

print("\n=== Validation Summary ===")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

print("\n=== Per-class AP@0.5 ===")
for i, name in enumerate(CLASS_NAMES):
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else float("nan")
    print(f"  {name:10s} ({CLASS_NAMES_AR[i]:>8s}): {ap50:.4f}")

## 6. Visualize sample predictions

In [ ]:
import random
from IPython.display import Image, display

val_images = list(val_dir.glob("*.jpg")) + list(val_dir.glob("*.png"))
samples = random.sample(val_images, min(6, len(val_images)))

preds = best.predict(source=[str(p) for p in samples], imgsz=640, conf=0.25, save=True,
                     project=str(WORK / "preview"), name="val_samples", exist_ok=True)

for r in preds:
    display(Image(r.save_dir + "/" + Path(r.path).name))

## 7. Export ONNX

ONNX lets us run inference without a PyTorch runtime in production — smaller Docker images, faster cold start.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
print(f"ONNX exported: {onnx_path}")

## 8. Stage artifacts for download

Copy `best.pt` + `best.onnx` + metrics plots into `/kaggle/working/artifacts/` so we can download them in one zip via the Kaggle UI (right sidebar → Output → Download).

In [ ]:
ART = WORK / "artifacts"
ART.mkdir(exist_ok=True)

# Weights
shutil.copy2(BEST_PT, ART / "best.pt")
shutil.copy2(Path(onnx_path), ART / "best.onnx")

# Metrics & plots
for plot in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
             "PR_curve.png", "F1_curve.png", "results.csv"]:
    src = RUN_DIR / plot
    if src.exists():
        shutil.copy2(src, ART / plot)

# Write a results.md summary
md = f"""# Training Results — YOLOv11s on X-Ray Baggage

| Metric | Value |
|---|---|
| mAP@0.5 | {metrics.box.map50:.4f} |
| mAP@0.5:0.95 | {metrics.box.map:.4f} |
| Precision | {metrics.box.mp:.4f} |
| Recall | {metrics.box.mr:.4f} |

## Per-class AP@0.5

| # | Class (EN) | Class (AR) | AP@0.5 |
|---|---|---|---|
"""
for i, name in enumerate(CLASS_NAMES):
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else float("nan")
    md += f"| {i} | {name} | {CLASS_NAMES_AR[i]} | {ap50:.4f} |\n"
(ART / "results.md").write_text(md, encoding="utf-8")

print("Artifacts ready:")
for f in sorted(ART.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1024:.1f} KB)")

## 9. Next steps

1. Download `best.pt` + `best.onnx` from Kaggle Output panel → place in `xRay_Detection/data/weights/` locally.
2. Copy `results.md` into the repo notebook folder.
3. Proceed to M1.2 — FastAPI inference service.